# QDM Map Alignment — Single

Align **one** target Bz map to a reference coordinate frame.

**Three-tier alignment:**

| Tier | Method | When |
|------|--------|------|
| 1 | ORB on raw LEDs | Always tried first |
| 2 | ORB with filter cycling (Sobel / Laplacian / Unsharp / CLAHE) | If tier 1 inliers < `MIN_INLIERS` |
| 3 | Manual tie-point clicking | If tier 2 also fails |

Place `alignment_functions.py` next to this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2
import os

from alignment_functions import (
    compute_affine_matrix,
    compute_affine_matrix_enhanced,
    compute_affine_matrix_manual,
    apply_affine,
    load_field_data,
    save_field_data,
)

---
## 1 — Configuration

In [ ]:
# ──────────────────────────────────────────────
# REFERENCE (coordinate frame you align TO)
# ──────────────────────────────────────────────
REFERENCE_LED = "./Testing_data/AF0/LED.jpg"

# ──────────────────────────────────────────────
# TARGET to align
# ──────────────────────────────────────────────
TARGET_LED   = "./Testing_data/AF7_5/LED.jpg"
TARGET_FIELD = "./Testing_data/AF7_5//Bz_uc0.mat"   # .mat | .npy | .csv


# ──────────────────────────────────────────────
# Output
# ──────────────────────────────────────────────
OUTPUT_DIR = "./aligned_output_single_alignment"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ──────────────────────────────────────────────
# Data loading options
# ──────────────────────────────────────────────
MAT_KEY   = "Bz"       # variable name inside .mat files
SCALE     = 1e9        # conversion factor (T -> nT); set 1.0 if already in nT
FLIPUD    = True       # flip Bz vertically (QDM convention)

# ──────────────────────────────────────────────
# Plot options
# ──────────────────────────────────────────────
PIXEL_SIZE_UM = 2.35   # QDM pixel size in um (e.g. 2.35); None = axes in pixels
VMIN = -10000           # colour scale min (e.g. -10000); None = data min
VMAX = 10000           # colour scale max (e.g.  10000); None = data max

# ──────────────────────────────────────────────
# Alignment tuning
# ──────────────────────────────────────────────
N_FEATURES   = 500     # ORB features to detect
N_MATCHES    = 100     # max matches for affine estimation
MIN_INLIERS  = 30      # below this, move to next tier

# Tier 2: enhanced filter cycling
ENHANCE_SIGMA   = 1.0  # blur sigma for Sobel, Laplacian, Unsharp filters
ENHANCE_FILTERS = ["sobel", "laplacian", "unsharp", "clahe"]

# Tier 3: manual tie-points (if both auto tiers fail)
N_TIE_POINTS = 6       # number of points to click (minimum 3)

# ──────────────────────────────────────────────
# Output format
# ──────────────────────────────────────────────
SAVE_EXT  = ".mat"     # ".mat" | ".npy" | ".csv"

---
## 2 — Run alignment (tiers 1 & 2, automatic)

In [ ]:
# Load images
ref_led = mpimg.imread(REFERENCE_LED)
tgt_led = mpimg.imread(TARGET_LED)

# Load Bz field
Bz = load_field_data(TARGET_FIELD, mat_key=MAT_KEY, scale=SCALE, flipud=FLIPUD)

# Resize LEDs to Bz resolution
bz_size = (Bz.shape[1], Bz.shape[0])
ref_led_r = cv2.resize(ref_led, bz_size, interpolation=cv2.INTER_AREA)
tgt_led_r = cv2.resize(tgt_led, bz_size, interpolation=cv2.INTER_AREA)

# ── Tier 1: ORB on raw LEDs ──
print("=" * 50)
print("  Tier 1: ORB on raw LEDs")
print("=" * 50)
M, aligned_led, n_inliers = compute_affine_matrix(
    ref_led_r, tgt_led_r,
    n_features=N_FEATURES, n_matches=N_MATCHES,
    sample_name="single", save_path=OUTPUT_DIR, show=True
)
method = "raw_ORB"

# ── Tier 2: enhanced filter cycling ──
if M is None or n_inliers < MIN_INLIERS:
    print(f"\nTier 1 gave {n_inliers} inliers (need {MIN_INLIERS})")
    print("=" * 50)
    print("  Tier 2: Enhanced ORB — cycling filters")
    print("=" * 50)
    M, aligned_led, n_inliers = compute_affine_matrix_enhanced(
        ref_led_r, tgt_led_r,
        sigma=ENHANCE_SIGMA,
        n_features=N_FEATURES, n_matches=N_MATCHES,
        min_inliers=MIN_INLIERS,
        filters=ENHANCE_FILTERS,
        sample_name="single_enhanced", save_path=OUTPUT_DIR, show=True
    )
    method = "enhanced_ORB"

# ── Apply if successful ──
if M is not None and n_inliers >= MIN_INLIERS:
    print(f"\nAlignment succeeded: {method}  ({n_inliers} inliers)")
    Bz_aligned = apply_affine(
        Bz, M, sample_name="single",
        vmin=VMIN, vmax=VMAX, pixel_size_um=PIXEL_SIZE_UM,
        save_path=OUTPUT_DIR, show=True
    )
    out_path = os.path.join(OUTPUT_DIR, f"Bz_aligned{SAVE_EXT}")
    save_field_data(out_path, Bz_aligned)
else:
    print(f"\nAutomatic alignment failed ({n_inliers} inliers).")
    print("Run the next cell for manual tie-point alignment (tier 3).")

---
## 3 — Manual tie-point alignment (tier 3, if needed)

Only run this cell if tiers 1 & 2 failed.

**Before running**, switch to an interactive matplotlib backend:

You will click corresponding points on the reference and target images.
After alignment, switch back with `%matplotlib inline`.

In [ ]:

# %matplotlib tk

# M, aligned_led, n_pts = compute_affine_matrix_manual(
#     ref_led_r, tgt_led_r,
#     n_points=N_TIE_POINTS,
#     sample_name="single_manual", save_path=OUTPUT_DIR, show=True
# )

# if M is not None:
#     print(f"\nManual alignment succeeded ({n_pts} tie-points)")

#     # Switch back to inline for the Bz plot
#     %matplotlib inline

#     Bz_aligned = apply_affine(
#         Bz, M, sample_name="single_manual",
#         vmin=VMIN, vmax=VMAX, pixel_size_um=PIXEL_SIZE_UM,
#         save_path=OUTPUT_DIR, show=True
#     )
#     out_path = os.path.join(OUTPUT_DIR, f"Bz_aligned{SAVE_EXT}")
#     save_field_data(out_path, Bz_aligned)
# else:
#     print("Manual alignment failed — not enough points picked.")